# Integrator RNN Model

[![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/brainpy/examples/blob/main/recurrent_networks/integrator_rnn.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/brainpy/examples/blob/main/recurrent_networks/integrator_rnn.ipynb)

In this notebook, we train a vanilla RNN to integrate white noise. This example is useful on its own to understand how RNN training works.

In [1]:
import matplotlib.pyplot as plt

import brainpy as bp
import brainpy.math as bm

bm.set_environment(bm.training_mode)

In [2]:
bp.__version__

'2.7.8'

## Parameters

In [3]:
dt = 0.04
num_step = int(1.0 / dt)
num_batch = 128

## Data

In [4]:
@bm.jit(static_argnames=['batch_size'])
def build_inputs_and_targets(mean=0.025, scale=0.01, batch_size=10):
  # Create the white noise input
  sample = bm.random.normal(size=(batch_size, 1, 1))
  bias = mean * 2.0 * (sample - 0.5)
  samples = bm.random.normal(size=(batch_size, num_step, 1))
  noise_t = scale / dt ** 0.5 * samples
  inputs = bias + noise_t
  targets = bm.cumsum(inputs, axis=1)
  return inputs, targets

In [5]:
def train_data():
  for _ in range(100):
    yield build_inputs_and_targets(batch_size=num_batch)

## Model

In [6]:
class RNN(bp.DynamicalSystem):
  def __init__(self, num_in, num_hidden):
    super(RNN, self).__init__()
    self.rnn = bp.dyn.RNNCell(num_in, num_hidden, train_state=True)
    self.out = bp.layers.Dense(num_hidden, 1)

  def update(self, x):
    return self.out(self.rnn(x))

model = RNN(1, 100)

## Training

In [7]:
# define loss function
def loss(predictions, targets, l2_reg=2e-4):
  mse = bp.losses.mean_squared_error(predictions, targets)
  l2 = l2_reg * bp.losses.l2_norm(model.train_vars().unique().dict()) ** 2
  return mse + l2


# define optimizer
lr = bp.optim.ExponentialDecay(lr=0.025, decay_steps=1, decay_rate=0.99975)
opt = bp.optim.Adam(lr=lr, eps=1e-1)


# create a trainer
trainer = bp.BPTT(model, loss_fun=loss, optimizer=opt)
trainer.fit(train_data,
            num_epoch=30,
            num_report=200)

/mnt/d/codes/projects/BrainPy/brainpy/optim/scheduler.py:376: UserWarning: ExponentialDecay is abandoned, please use ExponentialDecayLR insteadly.
  warnings.warn("ExponentialDecay is abandoned, please use ExponentialDecayLR insteadly.")


Train 200 steps, use 4.6996 s, loss 0.07158774882555008


Train 400 steps, use 4.2275 s, loss 0.020983107388019562


Train 600 steps, use 4.2556 s, loss 0.020277999341487885


Train 800 steps, use 4.2177 s, loss 0.019742079079151154


Train 1000 steps, use 4.2203 s, loss 0.0192656759172678


... (6 outputs left out)

Train 2400 steps, use 4.3525 s, loss 0.016519838944077492


Train 2600 steps, use 4.3519 s, loss 0.016221875324845314


Train 2800 steps, use 4.3311 s, loss 0.015903176739811897


Train 3000 steps, use 4.3609 s, loss 0.015615943819284439


In [8]:
plt.plot(bm.as_numpy(trainer.get_hist_metric()))
plt.show()

/tmp/ipykernel_141124/3451602226.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Testing

In [9]:
model.reset_state(1)
x, y = build_inputs_and_targets(batch_size=1)
predicts = trainer.predict(x)

plt.figure(figsize=(8, 2))
plt.plot(bm.as_numpy(y[0]).flatten(), label='Ground Truth')
plt.plot(bm.as_numpy(predicts[0]).flatten(), label='Prediction')
plt.legend()
plt.show()

  0%|          | 0/25 [00:00<?, ?it/s]

/tmp/ipykernel_141124/2137782508.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
